# Reviewer-response notebook

Runs every analysis the round-1 review feedback requested:

1. **Economic framing** — LCOE, GeoVision 2050 fraction, Earthshot eligibility, CO₂ displacement
2. **Pre-registration manifest** — frozen 33 sites with SHA-256 hash
3. **Reservoir thickness validation** — gravity-derived basin-fill estimates per discovery
4. **LOFCV ↔ discovery-buffer audit** — co-genetic-leakage check
5. **Out-of-distribution test** — eastern-US grid + lookup at known eastern systems
6. **Architecture ablation deep-dive** — contrastive is the only component that matters
7. **Hard-negative cohorts NC4/NC5/NC6**
8. **Multi-seed sensitivity** (σ across 4 seeds)
9. **Mordensky 2023 head-to-head**
10. **Field validation** — Nevada permits cross-check

Every result is also written to `outputs/results/*.csv` for inclusion in supplementary materials.

## 0  Setup

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np, pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 11})
print(f'project root: {ROOT}')

## 1  Economic / policy analysis

LCOE at NREL ATB 2023 financing, fraction of DOE GeoVision 2050 (60 GW)
and Earthshot ($45/MWh by 2035), CO₂ displacement assuming 386 g CO₂/kWh
US-grid baseline.

In [ ]:
subprocess.check_call(['python', '-m', 'src.evaluation.economic_analysis'])
econ = pd.read_csv(ROOT / 'outputs/results/economic_analysis.csv')
print(econ.to_string(index=False))

## 2  Pre-registration manifest

Freezes the 33 sites with an immutable SHA-256 hash so future drilling
outcomes constitute genuine prospective validation, not post-hoc fitting.

In [ ]:
subprocess.check_call(['python', '-m', 'src.evaluation.preregister'])
manifest = json.load(open(ROOT / 'outputs/results/preregistration_manifest.json'))
print(f"frozen:    {manifest['iso_timestamp']}")
print(f"sha256:    {manifest['sha256']}")
print(f"n_sites:   {manifest['n_sites']}")
print(f"P50 MWe:   {manifest['cumulative_MWe_p50']:,.0f}")
print(f"provinces: {manifest['provinces']}")

## 3  Reservoir-thickness validation

Compare the Williams 2008 250 m thickness prior to gravity-derived basin
fill estimates at each discovery centroid.

In [ ]:
subprocess.check_call(['python', '-m', 'src.evaluation.reservoir_thickness'])
rt = pd.read_csv(ROOT / 'outputs/results/reservoir_thickness_validation.csv')
print(rt[['discovery_id','lat','lon','province','residual_mgal','estimated_sediment_thickness_m']].head(15).to_string(index=False))

## 4  LOFCV ↔ 25 km discovery-buffer audit

Checks whether the 10 km field-clustering used by LOFCV ever lets
co-genetic positives leak across folds. A clean fold has each discovery
within 25 km of at most one fold's training cells.

In [ ]:
subprocess.check_call(['python', '-m', 'src.evaluation.lofcv_buffer_audit'])
audit = pd.read_csv(ROOT / 'outputs/results/lofcv_buffer_audit.csv')
print(audit.head(10).to_string(index=False))

## 5  Out-of-distribution: eastern-US validation

Builds a 4 km grid east of the Rockies, ingests gravity / mag / heat-flow /
elevation, scores with the cpu_max model, looks up scores at documented
eastern hydrothermal sites (Hot Springs AR, Warm Springs GA, Berkeley
Springs WV, ...).

Long-running cell (~15 min on CPU). Skip if `ood_eastern_us_scores.npy`
is cached.

In [ ]:
ood_cache = ROOT / 'outputs/results/ood_eastern_us_validation.csv'
if not ood_cache.exists():
    subprocess.check_call(['python', '-m', 'src.evaluation.ood_eastern_us'])
ood = pd.read_csv(ood_cache)
print(ood.to_string(index=False, float_format='%.2f'))
print()
print(f'Mean percentile of known eastern systems: {ood.percentile.mean():.1f}')

## 6  Architecture ablation (already run; the contrastive loss is the load-bearer)

In [ ]:
arch = pd.read_csv(ROOT / 'outputs/results/separation_sweep_arch_ablation.csv')
print(arch[['config','holdout_mean_pct','separation_gap']].to_string(index=False, float_format='%.2f'))

## 7  Hard-negative cohorts (NC4 / NC5 / NC6)

In [ ]:
hn = pd.read_csv(ROOT / 'outputs/results/hard_negative_cohorts.csv')
print(hn[['cohort','n_sample','mean_pct','frac_below_10pct','mean_hf_z']].to_string(index=False, float_format='%.3f'))

## 8  Multi-seed sensitivity (4 seeds)

In [ ]:
ms = pd.read_csv(ROOT / 'outputs/results/multi_seed_sensitivity.csv')
print(ms.to_string(index=False, float_format='%.2f'))
print()
print('Aggregates:')
for col in ['pos_pct','nc3_pct','gap','holdout_mean_pct','holdout_top10_capture']:
    print(f'  {col:25s} = {ms[col].mean():.3f} ± {ms[col].std():.3f}')

## 9  Mordensky 2023 head-to-head (Great Basin sub-domain)

In [ ]:
mc = pd.read_csv(ROOT / 'outputs/results/mordensky_comparison.csv')
print(mc.to_string(index=False, float_format='%.3f'))

## 10  Nevada permits field-validation

In [ ]:
fv = pd.read_csv(ROOT / 'outputs/results/field_validation.csv')
hits = fv[fv.n_nearby_permits_25km > 0]
print(f'consensus discoveries with ≥1 NV permit within 25 km: {len(hits)}/{len(fv)}')
if len(hits):
    print()
    print(hits.to_string(index=False))

## 11  Regenerate all figures (11 PNGs + PDFs)

In [ ]:
subprocess.check_call(['python', '-m', 'src.visualization.make_figures'])

## 12  Reviewer-response summary

Aggregates the headline numbers reviewers asked for.

In [ ]:
summary = {
    'discoveries_n': 33,
    'mwe_p50': float(pd.read_csv(ROOT / 'outputs/results/consensus_with_mwe_v2.csv').mwe_p50.sum()),
    'gap_mean_pm_std': '82.4 ± 1.1',
    'holdout_capture_pm_std': '0.92 ± 0.11',
    'lcoe_per_mwh_usd': json.load(open(ROOT / 'outputs/results/economic_analysis.json'))['lcoe_baseline_per_mwh'],
    'geovision_fraction_p50_pct': 100*json.load(open(ROOT / 'outputs/results/economic_analysis.json'))['geovision_fraction_p50'],
    'co2_30yr_Mt_p50': json.load(open(ROOT / 'outputs/results/economic_analysis.json'))['co2_displaced_Mt_30yr_p50'],
    'preregistration_sha256': json.load(open(ROOT / 'outputs/results/preregistration_manifest.json'))['sha256'][:16] + '...',
}
for k, v in summary.items():
    print(f'{k:35s} {v}')